**Last modified on**: 16/04/2026

**Author**: Onur Serçinoğlu

**Update Logs**:

16/04/2026: Initial version

**Credits**:

This Jupyter notebook accompanies the HMMER & Profile Hidden Markov Models lecture prepared for BENG451/BSB511 at Gebze Technical University. Key references:
- Eddy SR (1998). Profile hidden Markov models. Bioinformatics 14:755–763.
- Eddy SR (2011). Accelerated profile HMM searches. PLoS Comput Biol 7:e1002195.
- Durbin R et al. (1998). Biological sequence analysis. Cambridge University Press. (Chapter 5)
- Rabiner LR (1989). A tutorial on hidden Markov models. Proc IEEE 77:257–286.

# Hidden Markov Models and Profile HMMs

## Section 1: Motivation — The Remote Homology Problem

When we compare two protein sequences, their percent identity (PID) tells us roughly how similar they are. Empirical evidence has defined three zones:

| Zone | % Identity | Interpretation |
|------|-----------|----------------|
| **Safe zone** | > 30% | Homology is almost certainly real; structure/function likely conserved |
| **Twilight zone** | 20–30% | Homology probable but not guaranteed; use other evidence |
| **Midnight zone** | < 20% | Sequence identity no longer reliable; homology may still exist at the structural/functional level |

Standard BLAST uses **position-independent** scoring: every position in the alignment is scored the same way using a single substitution matrix (e.g., BLOSUM62). This works well when sequences are closely related, but fails in the midnight zone because:

1. The signal-to-noise ratio drops — chance alignments can score as well as real homologs.
2. **BLAST cannot capture that some positions are highly conserved across a family (e.g., catalytic residues) while others are nearly free to vary.** It assigns the same substitution cost to every position.

**Profile HMMs** solve this by building a statistical model from a *multiple sequence alignment* of known family members. Each position gets its own emission probabilities, capturing exactly which amino acids are tolerated at that site.

### 1.1 Position-specific vs. position-independent scoring: a numeric example

In [ ]:
import numpy as np

# ------------------------------------------------------------------
# Demonstration: position-independent (BLOSUM-style) vs. position-
# specific scoring for a single alignment column.
# ------------------------------------------------------------------

# Suppose the true background frequency of Glycine (G) is ~7%
background_frequency_G = 0.07

# Scenario A: Position-independent BLOSUM-style score
# The BLOSUM62 score for G->G is +6 (in half-bits).
# In log-odds terms: log2(observed/expected) ~ 6/2 = +3 bits
blosum62_GG_score = 6  # raw BLOSUM62 half-bit score

# Scenario B: Position-specific profile score at a CONSERVED position
# Suppose 70% of the family members have Gly at this position (strongly conserved).
family_frequency_G_conserved = 0.70  # 70% of the family has G here

# Log-odds score (in nats, base e) for observing G at this position:
# score = log(family_frequency / background_frequency)
profile_score_conserved = np.log(family_frequency_G_conserved / background_frequency_G)

# Scenario C: Position-specific profile score at a VARIABLE position
# Only 8% of family members have Gly here — close to background
family_frequency_G_variable = 0.08
profile_score_variable = np.log(family_frequency_G_variable / background_frequency_G)

print("=" * 55)
print("Position-independent vs. position-specific scoring")
print("=" * 55)
print()
print("Residue being scored: Glycine (G)")
print("Background frequency of G: %.2f" % background_frequency_G)
print()
print("[A] BLOSUM62 (position-independent) G->G score: %d half-bits" % blosum62_GG_score)
print("    This score is the SAME regardless of which position we are at.")
print()
print("[B] Profile score at a CONSERVED position (family freq = %.2f):" % family_frequency_G_conserved)
print("    log-odds score = log(%.2f / %.2f) = %.3f nats" % (
        family_frequency_G_conserved, background_frequency_G, profile_score_conserved))
print("    => STRONG reward for placing G here")
print()
print("[C] Profile score at a VARIABLE position (family freq = %.2f):" % family_frequency_G_variable)
print("    log-odds score = log(%.2f / %.2f) = %.3f nats" % (
        family_frequency_G_variable, background_frequency_G, profile_score_variable))
print("    => Nearly neutral — G is no more likely here than by chance")
print()
print("Conclusion: A profile captures position-specific conservation.")
print("BLAST cannot distinguish cases B and C — it always uses the same score.")

---

## Section 2: Markov Chains from First Principles

### The Markov property

A process satisfies the **Markov property** (is "memoryless") when the probability of the next state depends *only* on the current state, not on the history of previous states:

$$P(X_{t+1} = s \mid X_t, X_{t-1}, \ldots, X_0) = P(X_{t+1} = s \mid X_t)$$

**Weather analogy.** Suppose the weather can be in one of three states: **Sunny (S)**, **Cloudy (C)**, or **Rainy (R)**. If we assume that tomorrow's weather depends only on today's weather (and not on the whole week), the system is a Markov chain.

The transition probabilities can be represented as a matrix $T$ where $T_{ij} = P(\text{state } j \mid \text{state } i)$. Each row must sum to 1 (the chain must go *somewhere*).

In [ ]:
import numpy as np

# Weather states
states = ['Sunny', 'Cloudy', 'Rainy']
state_indices = {state: idx for idx, state in enumerate(states)}

# Transition matrix: rows = current state, columns = next state
# T[i][j] = P(next = states[j] | current = states[i])
transition_matrix = np.array([
    [0.7, 0.2, 0.1],   # from Sunny  -> [Sunny, Cloudy, Rainy]
    [0.4, 0.3, 0.3],   # from Cloudy -> [Sunny, Cloudy, Rainy]
    [0.2, 0.4, 0.4],   # from Rainy  -> [Sunny, Cloudy, Rainy]
])

print("Weather transition matrix")
print("-" * 40)
header = "%10s" % "" + "".join("%10s" % s for s in states)
print(header)
for i, row_state in enumerate(states):
    row_str = "%10s" % row_state
    for prob in transition_matrix[i]:
        row_str += "%10.2f" % prob
    print(row_str)

print()
print("Row sums (must all equal 1.0):", transition_matrix.sum(axis=1))

In [ ]:
import numpy as np

np.random.seed(42)

# Simulate a Markov chain for 30 steps, starting from 'Sunny'
num_steps = 30
current_state_index = state_indices['Sunny']

simulated_chain = [states[current_state_index]]

for step in range(num_steps - 1):
    row_probabilities = transition_matrix[current_state_index]
    next_state_index = np.random.choice(len(states), p=row_probabilities)
    simulated_chain.append(states[next_state_index])
    current_state_index = next_state_index

print("Simulated weather chain (30 steps, starting from Sunny):")
print()
for step_num, weather in enumerate(simulated_chain):
    print("  Day %2d: %s" % (step_num + 1, weather))

In [ ]:
import numpy as np

# Compute the probability of a specific 5-step sequence
# Example sequence: Sunny -> Cloudy -> Rainy -> Rainy -> Sunny
example_sequence = ['Sunny', 'Cloudy', 'Rainy', 'Rainy', 'Sunny']

# Assume the initial state probability (Sunny at day 1) = 1.0
sequence_probability = 1.0

print("Probability of the sequence:", " -> ".join(example_sequence))
print()

for step_idx in range(len(example_sequence) - 1):
    current = example_sequence[step_idx]
    next_state = example_sequence[step_idx + 1]
    current_idx = state_indices[current]
    next_idx = state_indices[next_state]
    transition_prob = transition_matrix[current_idx][next_idx]
    sequence_probability *= transition_prob
    print("  P(%s -> %s) = %.2f   running product = %.5f" % (
        current, next_state, transition_prob, sequence_probability))

print()
print("P(sequence) = %.6f" % sequence_probability)

**Question:** What does it mean for a process to be 'memoryless'? How is this useful for modeling sequences?

*Think about it:* If a process is memoryless, knowing the entire history of states before the current one adds no information about where it will go next. For biological sequences, this allows us to compute probabilities by multiplying simple two-state transition terms — making the math tractable. Without the Markov property, we would need to track exponentially many conditional probabilities (one for every possible history).

---

## Section 3: Hidden Markov Models

### From Markov chains to HMMs

In a plain Markov chain, the state is **observable** — we directly see whether today is Sunny, Cloudy, or Rainy.

In a **Hidden Markov Model (HMM)**, the states are **hidden** (latent). We only observe a sequence of *emissions* produced by the hidden states. The key insight:

- At each time step, the hidden state transitions to a new state (following a transition matrix, as before).
- Each hidden state *emits* an observable symbol with a state-specific probability (the **emission probability**).
- We observe only the emitted symbols, never the states directly.

In computational biology, the hidden states might represent "AT-rich regions" or "CG-rich regions" of a genome, and the emissions are the nucleotides A, T, C, G.

In [ ]:
import numpy as np

# ----------------------------------------------------------------
# A toy 2-state HMM: S1 = AT-rich region, S2 = CG-rich region
# ----------------------------------------------------------------

hmm_states = ['S1', 'S2']
nucleotides = ['A', 'T', 'C', 'G']

# Transition probabilities: P(next_state | current_state)
transition_probs = {
    'S1': {'S1': 0.7, 'S2': 0.3},
    'S2': {'S2': 0.6, 'S1': 0.4}
}

# Emission probabilities: P(nucleotide | state)
emission_probs = {
    'S1': {'A': 0.4, 'T': 0.4, 'C': 0.1, 'G': 0.1},  # AT-rich
    'S2': {'A': 0.1, 'T': 0.1, 'C': 0.4, 'G': 0.4}   # CG-rich
}

print("2-state HMM definition")
print("=" * 40)
print()
print("Transition probabilities:")
for from_state in hmm_states:
    for to_state in hmm_states:
        print("  P(%s -> %s) = %.1f" % (
            from_state, to_state, transition_probs[from_state][to_state]))
print()
print("Emission probabilities:")
for state in hmm_states:
    label = 'AT-rich' if state == 'S1' else 'CG-rich'
    print("  %s (%s): " % (state, label), end="")
    for nuc in nucleotides:
        print("%s=%.1f  " % (nuc, emission_probs[state][nuc]), end="")
    print()

In [ ]:
import numpy as np

np.random.seed(42)

def generate_hmm_sequence(transition_probs, emission_probs, hmm_states, nucleotides,
                           initial_state, sequence_length):
    """
    Generate a sequence from a 2-state HMM.
    Returns a tuple: (list of hidden states, string of observed nucleotides)
    """
    hidden_states_path = []
    observed_sequence = []

    current_state = initial_state

    for pos in range(sequence_length):
        hidden_states_path.append(current_state)

        # Emit a nucleotide from the current state
        emit_probs = [emission_probs[current_state][nuc] for nuc in nucleotides]
        emitted_nuc = np.random.choice(nucleotides, p=emit_probs)
        observed_sequence.append(emitted_nuc)

        # Transition to the next state
        trans_probs = [transition_probs[current_state][s] for s in hmm_states]
        current_state = np.random.choice(hmm_states, p=trans_probs)

    return hidden_states_path, ''.join(observed_sequence)


hidden_path, observed_seq = generate_hmm_sequence(
    transition_probs, emission_probs, hmm_states, nucleotides,
    initial_state='S1', sequence_length=25
)

print("Generated 25-nucleotide sequence from the HMM:")
print()
print("Position:   ", " ".join("%2d" % (i + 1) for i in range(25)))
print("Hidden state:", " ".join("%2s" % s for s in hidden_path))
print("Observed:   ", " ".join("%2s" % n for n in observed_seq))
print()
print("Observed sequence: ", observed_seq)

# Count nucleotides in S1 vs S2 regions to check the model makes sense
nuc_counts = {'S1': {'A': 0, 'T': 0, 'C': 0, 'G': 0},
              'S2': {'A': 0, 'T': 0, 'C': 0, 'G': 0}}
for state, nuc in zip(hidden_path, observed_seq):
    nuc_counts[state][nuc] += 1

print()
print("Nucleotide composition by hidden state (should reflect emission probs):")
for state in hmm_states:
    total = sum(nuc_counts[state].values())
    if total > 0:
        print("  %s: " % state, end="")
        for nuc in nucleotides:
            print("%s=%d(%.2f)  " % (
                nuc, nuc_counts[state][nuc], nuc_counts[state][nuc] / total), end="")
        print()

In [ ]:
import numpy as np
from itertools import product

# ---------------------------------------------------------------
# Brute-force computation of P(observation) for "ATCG"
# Sum over all 2^4 = 16 possible hidden state paths
# ---------------------------------------------------------------

observed_short = "ATCG"
obs_length = len(observed_short)  # 4
initial_state_prob = {'S1': 0.5, 'S2': 0.5}  # equal prior

total_probability = 0.0
path_contributions = []

# Enumerate all 2^4 = 16 state paths
for state_path in product(hmm_states, repeat=obs_length):
    path_prob = initial_state_prob[state_path[0]]

    # Multiply emission probability at position 0
    path_prob *= emission_probs[state_path[0]][observed_short[0]]

    # Multiply transition and emission probabilities for positions 1..3
    for pos in range(1, obs_length):
        prev_state = state_path[pos - 1]
        curr_state = state_path[pos]
        nucleotide = observed_short[pos]
        path_prob *= transition_probs[prev_state][curr_state]
        path_prob *= emission_probs[curr_state][nucleotide]

    total_probability += path_prob
    path_contributions.append((state_path, path_prob))

# Sort by contribution (highest first)
path_contributions.sort(key=lambda x: x[1], reverse=True)

print("Brute-force P(observation) for observed sequence: '%s'" % observed_short)
print("Summing over all %d possible hidden state paths:" % (len(hmm_states) ** obs_length))
print()
print("%-25s  %10s  %8s" % ("State path", "Probability", "% of total"))
print("-" * 50)

for state_path, path_prob in path_contributions:
    path_str = "->".join(state_path)
    percentage = 100.0 * path_prob / total_probability if total_probability > 0 else 0.0
    print("%-25s  %10.6f  %8.2f%%" % (path_str, path_prob, percentage))

print("-" * 50)
print("%-25s  %10.6f  %8.2f%%" % ("TOTAL P(ATCG)", total_probability, 100.0))
print()
best_path, best_prob = path_contributions[0]
print("Most probable path: %s  (probability = %.6f)" % ("->".join(best_path), best_prob))

### The Forward algorithm

The brute-force computation above evaluates every possible state path explicitly. For a sequence of length $L$ with $K$ hidden states, there are $K^L$ paths — exponential growth that becomes completely intractable even for short sequences.

The **Forward algorithm** computes the same sum $P(\text{observation})$ in $O(K^2 \cdot L)$ time using dynamic programming. It defines a forward variable:

$$\alpha_t(k) = P(o_1, o_2, \ldots, o_t, \text{state}_t = k)$$

and fills a $K \times L$ table by the recursion:

$$\alpha_t(k) = \left[ \sum_{j=1}^{K} \alpha_{t-1}(j) \cdot T_{jk} \right] \cdot e_k(o_t)$$

The final answer is $P(\text{observation}) = \sum_k \alpha_L(k)$. This converts an exponential sum into a polynomial one — exactly what makes HMMs computationally useful.

**Question:** In the brute-force calculation above, which state path contributed the most probability? Does that match your intuition given the emission probabilities?

*Hint:* Look at the observed sequence 'ATCG'. 'A' and 'T' are AT-rich emissions; 'C' and 'G' are CG-rich emissions. Which path assigns the right hidden state to each nucleotide?

---

## Section 4: Profile HMM Architecture (Plan7)

### From general HMMs to protein family models

For protein families, HMMER uses a specialized HMM architecture called **Plan7**. Unlike the general HMM above, a profile HMM is built from a multiple sequence alignment and has a fixed, biologically motivated structure.

At each consensus position $i$ (where $i = 1 \ldots L$, $L$ = alignment length), there are three state types:

| State | Symbol | Meaning |
|-------|--------|---------|
| **Match** | M_i | Position $i$ is occupied by an amino acid. Emission probabilities reflect which amino acids are tolerated here. |
| **Insert** | I_i | An insertion *after* position $i$ (extra residues not in the consensus). Typically models noise. |
| **Delete** | D_i | Position $i$ is skipped (a gap in the sequence at this column). No emission — silent state. |

### Why position-specific gap penalties?

In BLAST and pairwise alignment, a single gap-open penalty is used everywhere. In a profile HMM, the transition probabilities `MD` (match to delete) and `MI` (match to insert) are learned from the alignment — gaps are cheap where the family tolerates them (e.g., loop regions) and expensive at conserved structural positions.

### Comparison: BLAST vs. PSI-BLAST vs. Profile HMM

| Feature | BLAST | PSI-BLAST | Profile HMM |
|---------|-------|-----------|-------------|
| Residue scoring | Single substitution matrix (BLOSUM62) | Position-specific scoring matrix (PSSM) | Position-specific emission probabilities (log-odds) |
| Gap penalties | Uniform gap-open + gap-extend | Uniform gap-open + gap-extend | Position-specific (learned from alignment) |
| Family information | None — pairwise only | Multiple iterations against database | Full MSA-derived probability model |
| Sensitivity | Good for >40% identity | Better in twilight zone | Best — effective into midnight zone |

In [ ]:
import numpy as np

# -----------------------------------------------------------------------
# Define a toy 3-node profile HMM (3 consensus positions, L = 3)
# -----------------------------------------------------------------------

amino_acids = list('ACDEFGHIKLMNPQRSTVWY')  # 20 standard amino acids

# Emission probabilities for each match state
# Node 1: conserved — strongly prefers Glycine
# Node 2: hydrophobic cluster — prefers I, L, V, A, F, M
# Node 3: variable — nearly uniform distribution

def make_emission_dict(probs_list):
    """Create {aa: prob} dict from a list aligned to the amino_acids order."""
    return {aa: p for aa, p in zip(amino_acids, probs_list)}

# Node 1: conserved Gly position (Gly = index 5 in our aa list = 'G')
# G gets 0.70; remaining 0.30 spread thinly over other residues
# amino_acids = A C D E F G H I K L M N P Q R S T V W Y
#               0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
node1_raw = [0.01, 0.01, 0.01, 0.01, 0.01, 0.70,  # A C D E F G
             0.02, 0.02, 0.02, 0.02, 0.02, 0.01,  # H I K L M N
             0.01, 0.01, 0.01, 0.01, 0.01, 0.05,  # P Q R S T V
             0.01, 0.01]                           # W Y

# Node 2: hydrophobic — prefers I(7), L(9), V(17), A(0), F(4), M(10)
node2_raw = [0.12, 0.01, 0.01, 0.01, 0.10, 0.01,  # A C D E F G
             0.01, 0.15, 0.01, 0.18, 0.08, 0.01,  # H I K L M N
             0.01, 0.01, 0.01, 0.02, 0.02, 0.15,  # P Q R S T V
             0.05, 0.02]                           # W Y

# Node 3: variable — nearly uniform with slight preference for charged residues
base = 0.04
node3_raw = [base, base, 0.07, 0.08, base, base,  # A C D E F G
             base, base, 0.08, base, base, base,  # H I K L M N
             base, 0.06, 0.07, base, base, base,  # P Q R S T V
             base, base]                           # W Y

def normalize(raw_list):
    """Normalize a list so it sums to 1."""
    total = sum(raw_list)
    return [x / total for x in raw_list]

emission_probs_profile = {
    'M1': make_emission_dict(normalize(node1_raw)),
    'M2': make_emission_dict(normalize(node2_raw)),
    'M3': make_emission_dict(normalize(node3_raw)),
}

# Transition probabilities for each node
# Keys: MM (match->match), MI (match->insert), MD (match->delete)
#       IM (insert->match), II (insert->insert)
#       DM (delete->match), DD (delete->delete)
transition_probs_profile = {
    'M1': {'MM': 0.90, 'MI': 0.05, 'MD': 0.05,
           'IM': 0.70, 'II': 0.30,
           'DM': 0.90, 'DD': 0.10},
    'M2': {'MM': 0.85, 'MI': 0.05, 'MD': 0.10,
           'IM': 0.70, 'II': 0.30,
           'DM': 0.90, 'DD': 0.10},
    'M3': {'MM': 0.95, 'MI': 0.03, 'MD': 0.02,
           'IM': 0.80, 'II': 0.20,
           'DM': 0.95, 'DD': 0.05},
}

print("3-node profile HMM defined.")
print()
print("Top 5 emitted amino acids for each match state:")
print()
for node_name, em_dict in emission_probs_profile.items():
    sorted_aas = sorted(em_dict.items(), key=lambda x: x[1], reverse=True)
    top5 = sorted_aas[:5]
    top5_str = ", ".join("%s(%.3f)" % (aa, p) for aa, p in top5)
    print("  %s: %s" % (node_name, top5_str))

print()
print("Transition probabilities (MD = probability of skipping a position):")
for node_name, tr_dict in transition_probs_profile.items():
    print("  %s: MM=%.2f  MI=%.2f  MD=%.2f" % (
        node_name, tr_dict['MM'], tr_dict['MI'], tr_dict['MD']))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Plot emission probabilities for all 3 match states
node_labels = {
    'M1': 'Match state M1 (conserved)',
    'M2': 'Match state M2 (hydrophobic)',
    'M3': 'Match state M3 (variable)',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Profile HMM: Emission Probabilities for Each Match State', fontsize=13)

colors = ['steelblue', 'darkorange', 'forestgreen']

for ax, (node_name, title), color in zip(axes, node_labels.items(), colors):
    probs = [emission_probs_profile[node_name][aa] for aa in amino_acids]
    ax.bar(amino_acids, probs, color=color, edgecolor='black', linewidth=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Amino acid (single letter code)', fontsize=10)
    ax.set_ylabel('Emission probability', fontsize=10)
    ax.set_ylim(0, 0.8)
    ax.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.show()

**Question:** Which match state looks like a position buried in the hydrophobic core? How can you tell from the emission probabilities?

*Hint:* Residues buried in the hydrophobic core of a protein are almost exclusively hydrophobic: Ile (I), Leu (L), Val (V), Ala (A), Phe (F), Met (M). A position where only these residues have high emission probabilities is likely buried and structurally important.

---

## Section 5: Scoring — Viterbi vs. Forward

### Two fundamental algorithms

Given a sequence and a profile HMM, we want to compute a score. Two algorithms answer slightly different questions:

| Algorithm | Question answered | Analogy |
|-----------|-------------------|----------|
| **Viterbi** | What is the probability of the single *best* path through the model? | Smith-Waterman DP — find the optimal alignment |
| **Forward** | What is the total probability summed over *all* paths? | Sum of all alignments, weighted by probability |

### Key relationship

- **Forward score ≥ Viterbi score** always (the Viterbi path is one of many paths that Forward sums over).
- For **close homologs**: almost all the probability is concentrated in one dominant path → Forward ≈ Viterbi.
- For **remote homologs**: probability is spread over many sub-optimal paths → Forward >> Viterbi. Using only Viterbi would underestimate the true likelihood of homology.

This is why HMMER uses the Forward score (not Viterbi) for E-value computation — it is more sensitive for remote homologs.

In [ ]:
import numpy as np

# ---------------------------------------------------------------
# Viterbi and Forward algorithms as proper DP tables
# on the toy 3-node profile HMM from Section 4, query "GLA"
#
# For clarity we restrict to the 3 match states only (no inserts/
# deletes). Each algorithm fills a K x L table where:
#   K = number of states  (here K=3: M1, M2, M3)
#   L = query length      (here L=3: G, L, A)
# ---------------------------------------------------------------

query_peptide = list("GLA")          # ['G', 'L', 'A']
states        = ['M1', 'M2', 'M3']  # K = 3
K = len(states)
L = len(query_peptide)

# Background amino acid frequencies (approximate, from UniProtKB)
background_freq = {
    'A': 0.074, 'C': 0.025, 'D': 0.054, 'E': 0.054,
    'F': 0.047, 'G': 0.074, 'H': 0.026, 'I': 0.068,
    'K': 0.058, 'L': 0.099, 'M': 0.025, 'N': 0.045,
    'P': 0.039, 'Q': 0.034, 'R': 0.052, 'S': 0.057,
    'T': 0.051, 'V': 0.073, 'W': 0.013, 'Y': 0.032
}

# Pre-compute log-odds emission score for each (residue, state) pair
def log_odds(residue, state):
    """log( P(residue | state) / P(residue | background) )"""
    return np.log(emission_probs_profile[state][residue] /
                  background_freq[residue])

# Transition: log P(M_i -> M_{i+1})  (match-to-match only, for this demo)
def log_trans(from_state):
    return np.log(transition_probs_profile[from_state]['MM'])


# ================================================================
# Viterbi algorithm
# ================================================================
def viterbi(query, states, log_odds_fn, log_trans_fn):
    """
    Viterbi algorithm: finds the single most probable path.

    Table dp[k][l] = best cumulative log-odds score when the
    query position l is aligned to state k (states[k]).

    Returns: (best_score, traceback_path_list)
    """
    K = len(states)
    L = len(query)

    # dp[k][l]        — best log-odds score reaching state k at position l
    # tb[k][l]        — index of the best predecessor state
    dp = np.full((K, L), -np.inf)
    tb = np.full((K, L), -1, dtype=int)

    # ---- Initialisation: position 0, only state 0 (M1) is reachable ----
    dp[0][0] = log_odds_fn(query[0], states[0])

    print("Viterbi DP table  (rows = states, columns = query positions)")
    print("=" * 65)
    print("Initialisation  (position 0, residue '%s'):" % query[0])
    print("  dp[M1][0] = log_odds('%s', M1) = %.4f" % (query[0], dp[0][0]))
    print("  dp[M2][0] = -inf  (M2 unreachable at position 0 in match-only model)")
    print("  dp[M3][0] = -inf  (M3 unreachable at position 0 in match-only model)")
    print()

    # ---- Recursion: fill left to right, top to bottom ----
    print("Recursion:")
    for l in range(1, L):
        residue = query[l]
        for k in range(l, K):          # state index must be >= position index
            best_prev_score = -np.inf
            best_prev_k     = -1
            for j in range(k):         # only the immediately preceding state
                if j != k - 1:        # match-only: only k-1 -> k is valid
                    continue
                candidate = dp[j][l-1] + log_trans_fn(states[j])
                if candidate > best_prev_score:
                    best_prev_score = candidate
                    best_prev_k     = j
            if best_prev_score > -np.inf:
                dp[k][l] = best_prev_score + log_odds_fn(residue, states[k])
                tb[k][l] = best_prev_k
                print("  dp[%s][%d] = dp[%s][%d] + log_trans(%s->%s) + log_odds('%s','%s')"
                      % (states[k], l, states[j], l-1,
                         states[j], states[k], residue, states[k]))
                print("           = %.4f + %.4f + %.4f = %.4f"
                      % (dp[j][l-1], log_trans_fn(states[j]),
                         log_odds_fn(residue, states[k]), dp[k][l]))
    print()

    # ---- Termination: best score is at the last state, last position ----
    best_k     = int(np.argmax(dp[:, L-1]))
    best_score = dp[best_k][L-1]

    # ---- Traceback ----
    path_indices = [best_k]
    for l in range(L-1, 0, -1):
        best_k = tb[best_k][l]
        path_indices.insert(0, best_k)
    path = [states[k] for k in path_indices]

    print("Termination:")
    print("  Best final state : %s  (position %d)" % (states[int(np.argmax(dp[:, L-1]))], L-1))
    print("  Viterbi score    : %.4f" % best_score)
    print()
    print("Traceback path: %s" % " -> ".join(
        "%s(%s)" % (s, r) for s, r in zip(path, query)))
    print()

    # Pretty-print the full DP table
    print("Full Viterbi DP table:")
    header = "       " + "".join("  pos%d(%s)" % (l, query[l]) for l in range(L))
    print(header)
    for k, state in enumerate(states):
        row = "  %s  " % state
        for l in range(L):
            v = dp[k][l]
            row += "  %9.4f" % v if v > -np.inf else "       -inf"
        print(row)
    print()

    return best_score, path


# ================================================================
# Forward algorithm
# ================================================================
def forward(query, states, log_odds_fn, log_trans_fn):
    """
    Forward algorithm: sums probabilities over ALL paths.

    Table dp[k][l] = log( sum of probabilities over all paths
                          that end at state k having consumed
                          query[0..l] ).

    Uses the log-sum-exp trick:
        log(exp(a) + exp(b)) = np.logaddexp(a, b)
    which avoids floating-point underflow when working in log space.

    Key insight: forward_score >= viterbi_score always.
    """
    K = len(states)
    L = len(query)

    dp = np.full((K, L), -np.inf)

    # ---- Initialisation ----
    dp[0][0] = log_odds_fn(query[0], states[0])

    print("Forward DP table  (rows = states, columns = query positions)")
    print("=" * 65)
    print("Initialisation  (position 0, residue '%s'):" % query[0])
    print("  dp[M1][0] = log_odds('%s', M1) = %.4f" % (query[0], dp[0][0]))
    print("  dp[M2][0] = -inf")
    print("  dp[M3][0] = -inf")
    print()

    # ---- Recursion ----
    print("Recursion  (using log-sum-exp to accumulate ALL predecessor paths):")
    for l in range(1, L):
        residue = query[l]
        for k in range(l, K):
            # Accumulate over all valid predecessor states j
            accumulated = -np.inf
            for j in range(k):
                if j != k - 1:
                    continue
                candidate = dp[j][l-1] + log_trans_fn(states[j])
                accumulated = np.logaddexp(accumulated, candidate)
            if accumulated > -np.inf:
                dp[k][l] = accumulated + log_odds_fn(residue, states[k])
                print("  dp[%s][%d] = logaddexp(predecessors) + log_odds('%s','%s')"
                      % (states[k], l, residue, states[k]))
                print("           = %.4f + %.4f = %.4f"
                      % (accumulated, log_odds_fn(residue, states[k]), dp[k][l]))
    print()

    # ---- Termination: log-sum-exp over all final states ----
    forward_score = -np.inf
    for k in range(K):
        forward_score = np.logaddexp(forward_score, dp[k][L-1])

    print("Termination  (log-sum-exp over all states at final position):")
    for k, state in enumerate(states):
        v = dp[k][L-1]
        print("  dp[%s][%d] = %s" % (state, L-1,
              "%.4f" % v if v > -np.inf else "-inf"))
    print("  Forward score = logaddexp(...) = %.4f" % forward_score)
    print()

    # Pretty-print the full DP table
    print("Full Forward DP table:")
    header = "       " + "".join("  pos%d(%s)" % (l, query[l]) for l in range(L))
    print(header)
    for k, state in enumerate(states):
        row = "  %s  " % state
        for l in range(L):
            v = dp[k][l]
            row += "  %9.4f" % v if v > -np.inf else "       -inf"
        print(row)
    print()

    return forward_score


# ================================================================
# Run both algorithms and compare
# ================================================================
print("Query peptide : '%s'" % "".join(query_peptide))
print("Profile states: %s" % ", ".join(states))
print()
print("=" * 65)
print("VITERBI")
print("=" * 65)
viterbi_score, best_path = viterbi(query_peptide, states, log_odds, log_trans)

print("=" * 65)
print("FORWARD")
print("=" * 65)
forward_score = forward(query_peptide, states, log_odds, log_trans)

print("=" * 65)
print("COMPARISON SUMMARY")
print("=" * 65)
print()
print("  Viterbi score (best single path) : %8.4f" % viterbi_score)
print("  Forward score (sum over all paths): %8.4f" % forward_score)
print("  Difference (Forward - Viterbi)    : %8.4f" % (forward_score - viterbi_score))
print()
print("  Best Viterbi path  : %s" % " -> ".join(
      "%s(%s)" % (s, r) for s, r in zip(best_path, query_peptide)))
print()
print("Interpretation:")
print("  The Viterbi score is the log-odds of the single best alignment.")
print("  The Forward score accounts for ALL possible alignments; because")
print("  the match-only path dominates for this close-homolog peptide,")
print("  the two scores are very similar here.")
print("  For remote homologs, many sub-optimal paths contribute and")
print("  Forward >> Viterbi, making Forward more sensitive.")

In [ ]:
import numpy as np

# ---------------------------------------------------------------
# Illustrate: Viterbi ≈ Forward for close homologs,
#             Forward >> Viterbi for remote homologs
#
# These values are representative of what HMMER outputs in practice
# for a globin family model searched against:
#   - a known globin (close homolog)
#   - a distantly related oxygen-binding protein (remote homolog)
#   - a random sequence (no homology)
# ---------------------------------------------------------------

examples = [
    ("Close homolog (e.g., human hemoglobin beta)",  28.5,  29.1,  "strong homolog"),
    ("Remote homolog (e.g., sea cucumber globin)",    8.2,  14.7,  "remote homolog"),
    ("Random sequence",                              -3.1,  -0.8,  "noise"),
]

print("Viterbi vs. Forward scores: practical comparison")
print("=" * 75)
print()
print("%-42s  %9s  %9s  %9s  %s" % (
    "Sequence", "Viterbi", "Forward", "Fwd-Vit", "Interpretation"))
print("-" * 90)

for label, vit, fwd, interp in examples:
    diff = fwd - vit
    print("%-42s  %9.1f  %9.1f  %9.1f  %s" % (label, vit, fwd, diff, interp))

print()
print("Key observation:")
print("  - For the close homolog, Viterbi and Forward are nearly identical.")
print("    One dominant alignment path captures most of the probability.")
print("  - For the remote homolog, Forward >> Viterbi.")
print("    Many sub-optimal alignment paths together contribute significant")
print("    probability that Viterbi ignores.")
print("  - HMMER uses Forward scores (not Viterbi) for E-value computation.")

**Question:** If Forward score is always ≥ Viterbi, why would you ever use Viterbi? (Hint: think about what information each algorithm provides.)

*Answer hint:* Viterbi returns not just a score but also the **single best alignment path** — i.e., the domain annotation telling you exactly which residues in the query match which positions in the profile HMM. This is how HMMER produces the aligned output you see in `.sto` or tabular outputs. Forward only tells you the total probability; it does not tell you where the domain boundaries are or how the sequence threads through the model.

---

## Section 6: E-values and Statistical Significance

### What is an E-value?

A raw score tells us how well a sequence matched a profile, but scores are not directly comparable across different searches (different database sizes, different model lengths). We need a statistic that accounts for chance.

The **E-value** (Expect value) is defined as:

$$E = N \times P(\text{score} \geq s \mid \text{null model})$$

where $N$ is the number of sequences in the database and $P(\text{score} \geq s \mid \text{null})$ is the probability of achieving score $s$ or better by chance against a random sequence.

Interpretation: an E-value of 0.001 means we expect to see 0.001 sequences scoring this well **by chance** in a database of size $N$. Small E-values = unlikely to be chance = likely real homology.

### Typical thresholds

HMMER's default inclusion threshold is **E < 0.01**. Many researchers use stricter cutoffs:

| E-value range | Interpretation |
|--------------|----------------|
| E > 1 | Likely noise — expect multiple false positives |
| 0.01 < E ≤ 1 | Borderline — inspect alignment manually |
| 1×10⁻⁵ < E ≤ 0.01 | Significant homolog — probably real |
| E ≤ 1×10⁻⁵ | Strong hit — high confidence homology |

In [ ]:
import numpy as np

# E-value interpretation table as printed output
print("E-value ranges and their interpretations")
print("=" * 65)
print()
print("%-25s  %-20s  %s" % ("E-value range", "Expected FP count", "Interpretation"))
print("-" * 65)

evalue_table = [
    ("E > 1",              "≥ 1",          "Likely noise"),
    ("0.01 < E ≤ 1",       "0.01 – 1",     "Borderline; check manually"),
    ("1e-5 < E ≤ 0.01",    "0.00001 – 0.01", "Significant homolog"),
    ("E ≤ 1e-5",           "< 0.00001",    "Strong hit"),
]

for erange, fpcount, interp in evalue_table:
    print("%-25s  %-20s  %s" % (erange, fpcount, interp))

print()
print("Note: E-value = N * P(score >= s | null model)")
print("      where N = database size")

In [ ]:
import numpy as np

# ---------------------------------------------------------------
# Null model demonstration:
# Compare log-odds scores of a real globin fragment vs. a random
# sequence against the 3-node profile HMM.
#
# The null model in HMMER assumes each position is drawn from
# the background amino acid frequencies (null2 model).
# A random sequence should score near 0 log-odds on average.
# ---------------------------------------------------------------

np.random.seed(42)

# Real globin-like fragment — Gly at conserved pos, Leu at hydrophobic pos
globin_fragment = "GLA"

# Generate a random peptide of the same length from background frequencies
aa_list_bg = list(background_freq.keys())
bg_prob_list = [background_freq[aa] for aa in aa_list_bg]
random_peptide_residues = np.random.choice(aa_list_bg, size=3, p=bg_prob_list)
random_peptide = ''.join(random_peptide_residues)

def score_fully_matched_path(peptide, profile_nodes, emission_probs_profile,
                              transition_probs_profile, background_freq):
    """Compute the log-odds score for a peptide aligned to all match states."""
    if len(peptide) != len(profile_nodes):
        return float('-inf')
    total_score = 0.0
    for idx, (residue, node) in enumerate(zip(peptide, profile_nodes)):
        emission_p = emission_probs_profile[node][residue]
        bg_p = background_freq[residue]
        total_score += np.log(emission_p / bg_p)
        if idx < len(profile_nodes) - 1:
            total_score += np.log(transition_probs_profile[node]['MM'])
    return total_score

globin_score = score_fully_matched_path(
    globin_fragment, profile_nodes,
    emission_probs_profile, transition_probs_profile, background_freq)

random_score = score_fully_matched_path(
    random_peptide, profile_nodes,
    emission_probs_profile, transition_probs_profile, background_freq)

print("Null model demonstration")
print("=" * 50)
print()
print("Profile HMM: 3-node toy globin-like model")
print()
print("Query 1 (real globin fragment): '%s'" % globin_fragment)
print("  All-match-state log-odds score: %.4f" % globin_score)
print("  Interpretation: positive score -> more likely from the family than from noise")
print()
print("Query 2 (random sequence):       '%s'" % random_peptide)
print("  All-match-state log-odds score: %.4f" % random_score)
print("  Interpretation: near-zero score -> consistent with random sequence")
print()
print("Score difference: %.4f" % (globin_score - random_score))
print()
print("A positive score >> 0 indicates the query is more consistent with")
print("the family model than with background (null) random sequences.")

**Question:** If you search a database of 1 million sequences and get a hit with E-value = 0.05, how many false positives would you expect? Is this acceptable?

*Answer:* E-value = 0.05 means you expect **0.05 false positives** on average in that search. In other words, if you searched 20 independent databases of 1 million sequences each and the true hit has no homology, you would expect about 1 false positive across all 20 searches. For a one-off search, a single hit with E = 0.05 is borderline — worth inspecting the alignment manually but not automatically accepted. If you have many hits at this level, most are likely noise. HMMER's default cutoff of 0.01 is more conservative for exactly this reason.

---

## Section 7: Resources

| Resource | Type | URL / Reference |
|----------|------|-----------------|
| HMMER User Guide | Documentation | http://hmmer.org/documentation.html |
| Eddy 1998 — Profile HMMs | Paper | Bioinformatics 14:755–763 |
| Eddy 2011 — HMMER3 | Paper | PLoS Comput Biol 7:e1002195 |
| Durbin et al. 1998 | Textbook | Chapter 5 covers HMM theory in full |
| Rabiner 1989 Tutorial | Tutorial paper | Proc IEEE 77:257–286 |